In [1]:
import os
import shutil
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Loading the dataset from google drive

base_dir = '/content/drive/MyDrive/dataset'


train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

# 2. Reorganization

fruits_list = ['banana', 'apple', 'pear', 'grapes', 'orange', 'kiwi', 'watermelon', 'pomegranate', 'pineapple', 'mango']
vegetables_list = ['cucumber', 'carrot', 'capsicum', 'onion', 'potato', 'lemon', 'tomato', 'radish', 'raddish', 'beetroot', 'cabbage',
                   'lettuce', 'spinach', 'soybean', 'soy beans', 'cauliflower', 'bell pepper', 'chilli pepper', 'chilly', 'pepper', 'turnip',
                   'corn', 'sweetcorn', 'sweet potato', 'sweetpotato', 'paprika', 'jalapeño', 'jalepeno', 'ginger', 'garlic', 'peas', 'eggplant']

def organize_binary_classes(path):
    if not os.path.exists(path):
        print(f"Directory {path} not found!")
        return

    fruits_target = os.path.join(path, 'Fruits')
    veg_target = os.path.join(path, 'Vegetables')

    # Check if already reorganized
    if os.path.exists(fruits_target):
        print(f"Folder {path} already reorganized. Skipping.")
        return

    os.makedirs(fruits_target, exist_ok=True)
    os.makedirs(veg_target, exist_ok=True)

    for folder_name in os.listdir(path):
        folder_path = os.path.join(path, folder_name)
        if not os.path.isdir(folder_path) or folder_name in ['Fruits', 'Vegetables']:
            continue

        folder_lower = folder_name.lower().strip()
        dest = fruits_target if folder_lower in fruits_list else veg_target if folder_lower in vegetables_list else None

        if dest:
            for img in os.listdir(folder_path):
                shutil.move(os.path.join(folder_path, img), os.path.join(dest, f"{folder_name}_{img}"))
            os.rmdir(folder_path)

# Run the reorganization
print("Reorganizing folders...")
organize_binary_classes(train_dir)
organize_binary_classes(val_dir)
organize_binary_classes(test_dir)


# 3. Load data and build model

img_height, img_width = 150, 150
batch_size = 32

print("\nLoading datasets...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=(img_height, img_width), batch_size=batch_size, label_mode='binary'
)
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=(img_height, img_width), batch_size=batch_size, label_mode='binary'
)
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=(img_height, img_width), batch_size=batch_size, label_mode='binary'
)

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\nStarting Training...")
model.fit(train_dataset, validation_data=validation_dataset, epochs=10)

# 4. Evaluation
test_loss, test_acc = model.evaluate(test_dataset)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")
model.save('fruit_veg_model.keras')

Reorganizing folders...
Folder /content/drive/MyDrive/dataset/train already reorganized. Skipping.
Folder /content/drive/MyDrive/dataset/validation already reorganized. Skipping.
Folder /content/drive/MyDrive/dataset/test already reorganized. Skipping.

Loading datasets...
Found 3115 files belonging to 2 classes.
Found 351 files belonging to 2 classes.
Found 359 files belonging to 2 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Starting Training...
Epoch 1/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 397s 4s/step - accuracy: 0.7159 - loss: 0.7897 - val_accuracy: 0.7322 - val_loss: 0.5618
Epoch 2/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - accuracy: 0.7374 - loss: 0.5343 - val_accuracy: 0.7407 - val_loss: 0.5350
Epoch 3/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 123s 1s/step - accuracy: 0.7727 - loss: 0.4713 - val_accuracy: 0.8034 - val_loss: 0.3793
Epoch 4/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 149s 1s/step - accuracy: 0.8324 - loss: 0.3807 - val_accuracy: 0.9117 - val_loss: 0.2996
Epoch 5/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 151s 1s/step - accuracy: 0.8838 - loss: 0.2768 - val_accuracy: 0.9630 - val_loss: 0.1586
Epoch 6/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 144s 1s/step - accuracy: 0.9268 - loss: 0.1827 - val_accuracy: 0.9430 - val_loss: 0.1616
Epoch 7/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 139s 1s/step - accuracy: 0.9583 - loss: 0.1198 - val_accuracy: 0.9573 - val_loss: 0.1136
Epoch 8/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 148s 2s/step - accuracy: 0.9647 - loss: 0.0968 - va

In [4]:
import tensorflow as tf
import numpy as np
import os

# 1. Load the saved model
model_path = 'fruit_veg_model.keras'

if not os.path.exists(model_path):
    print(f"Error: Could not find '{model_path}'. Make sure you trained and saved the model first!")
    exit()

print("Loading model...")
model = tf.keras.models.load_model(model_path)



image_path = r"/content/cucumber.jpg"

if not os.path.exists(image_path):
    print(f"Error: Could not find the image '{image_path}'. Please check the path.")
    exit()

# 3. Preprocess the image to match what the model expects

img = tf.keras.utils.load_img(image_path, target_size=(150, 150))
img_array = tf.keras.utils.img_to_array(img)


img_array = tf.expand_dims(img_array, 0)

# 4. Make the prediction
print("Classifying image...")
predictions = model.predict(img_array)


score = predictions[0][0]

if score < 0.5:
    class_name = "Fruit"
    confidence = (1 - score) * 100
else:
    class_name = "Vegetable"
    confidence = score * 100

print(f"Prediction: {class_name}")
print(f"Confidence: {confidence:.2f}%")

Loading model...
Classifying image...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
Prediction: Vegetable
Confidence: 91.07%
